# MCS603 — Data Analysis Module
## Notebook 5: Streamlit — Interactive Health Data Dashboard

---
### Learning Objectives
- Understand the Streamlit execution model
- Build interactive widgets: sliders, dropdowns, checkboxes, multiselect
- Display charts (Matplotlib, Seaborn, Plotly) in a web app
- Use `st.cache_data` for performance
- Lay out a multi-section dashboard with columns and tabs
- Deploy locally and understand cloud deployment options

### Setup
```bash
pip install streamlit plotly pandas scipy matplotlib seaborn
streamlit run dashboard.py
```

### Outline
1. Streamlit Fundamentals  
2. Widgets Reference  
3. Layout Primitives  
4. Charting in Streamlit  
5. The Complete Dashboard (`dashboard.py`)  
6. Running & Deployment  
7. Exercises  

---
## 1. Streamlit Fundamentals

Streamlit turns a plain Python script into a web app.  
**Every time a widget changes, the script runs top-to-bottom.**  
State is managed via `st.session_state` and caching.

```python
import streamlit as st

# Title / text
st.title("My App")
st.header("Section")
st.markdown("**bold** and _italic_")

# Metrics
st.metric("Life Expectancy", "63.2 yrs", delta="+1.2 yrs")

# Data
st.dataframe(df)
st.table(df.head())

# Caching — prevents re-running expensive functions
@st.cache_data
def load_data():
    return pd.read_csv("africa_health_clean.csv")
```

---
## 2. Widgets Reference

| Widget | Code | Returns |
|---|---|---|
| Text input | `st.text_input("Name")` | `str` |
| Number input | `st.number_input("Age", min_value=0)` | `float` |
| Slider | `st.slider("Year", 2000, 2025, 2022)` | value |
| Range slider | `st.slider("Range", 0, 100, (20, 80))` | `tuple` |
| Selectbox | `st.selectbox("Region", options)` | selected item |
| Multiselect | `st.multiselect("Columns", cols)` | `list` |
| Checkbox | `st.checkbox("Show raw data")` | `bool` |
| Radio | `st.radio("Chart type", ["Bar","Line"])` | selected |
| Button | `st.button("Run Analysis")` | `bool` (one shot) |
| File uploader | `st.file_uploader("Upload CSV")` | file object |
| Date picker | `st.date_input("Date")` | `date` |

**Sidebar widgets** use `st.sidebar.widget(...)` — same API.

---
## 3. Layout Primitives

```python
# Columns
col1, col2, col3 = st.columns([2, 1, 1])   # proportional widths
with col1:
    st.metric("Life Expectancy", "63.2")

# Tabs
tab1, tab2 = st.tabs(["Overview", "Details"])
with tab1:
    st.write("Overview content")

# Expander
with st.expander("Show raw data"):
    st.dataframe(df)

# Container
with st.container():
    st.write("Grouped content")

# Sidebar
with st.sidebar:
    region = st.selectbox("Region", options)
```

---
## 4. Charting in Streamlit

| Method | Library | Notes |
|---|---|---|
| `st.line_chart(df)` | Built-in | Quick, limited customisation |
| `st.bar_chart(df)` | Built-in | Quick bar chart |
| `st.pyplot(fig)` | Matplotlib | Full Matplotlib control |
| `st.plotly_chart(fig)` | Plotly | Interactive (zoom, hover, pan) |
| `st.altair_chart(c)` | Altair | Declarative, interactive |

---
## 5. The Complete Dashboard

Run the cell below to write `dashboard.py` to disk, then launch it with `streamlit run dashboard.py`.

In [ ]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from scipy.stats.mstats import winsorize
import matplotlib.gridspec as gridspec

# ── Page config ────────────────────────────────────────────────────
st.set_page_config(
    page_title="Africa Health Dashboard",
    page_icon="🌍",
    layout="wide",
    initial_sidebar_state="expanded",
)

REGION_COLORS = {
    "West Africa":    "#E74C3C",
    "East Africa":    "#3498DB",
    "North Africa":   "#2ECC71",
    "Central Africa": "#9B59B6",
    "Southern Africa":"#F39C12",
}

# ── Data ────────────────────────────────────────────────────────────
@st.cache_data
def generate_data():
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d\'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63,8,N),45,80),
        "infant_mortality"  : np.clip(np.random.exponential(35,N),5,120),
        "maternal_mortality": np.clip(np.random.exponential(350,N),20,2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5,N),0.1,28),
        "health_expenditure": np.clip(np.random.normal(5.2,2.1,N),1,12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2,1.2,N)),300,15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78,18,N),20,99),
        "water_access"      : np.clip(np.random.normal(68,22,N),15,99),
        "malaria_incidence" : np.clip(np.random.exponential(120,N),0,450),
        "physicians_per1k"  : np.clip(np.random.exponential(0.8,N),0.02,5),
    })
    # Impute NaNs with region median
    for col in ["maternal_mortality","physicians_per1k","malaria_incidence"]:
        idx = np.random.choice(df.index,size=int(0.12*N),replace=False)
        df.loc[idx,col] = np.nan
        df[col] = df.groupby("region")[col].transform(lambda x: x.fillna(x.median()))
    # Feature engineering
    def norm(s): return (s - s.min()) / (s.max() - s.min())
    df["health_index"] = (
        norm(df["life_expectancy"])*0.35 + norm(df["vaccination_dtp3"])*0.25 +
        norm(df["water_access"])*0.20 + (1-norm(df["infant_mortality"]))*0.20
    ) * 100
    df["disease_burden"] = (
        norm(df["infant_mortality"])*0.35 + norm(df["maternal_mortality"])*0.30 +
        norm(df["hiv_prevalence"])*0.20 + norm(df["malaria_incidence"])*0.15
    ) * 100
    df["gdp_band"] = pd.qcut(df["gdp_per_capita"],q=4,labels=["Low","Lower-Mid","Upper-Mid","High"])
    return df

df_full = generate_data()

# ── Sidebar ──────────────────────────────────────────────────────────
st.sidebar.image("https://upload.wikimedia.org/wikipedia/commons/thumb/4/4f/Africa_%28orthographic_projection%29.svg/200px-Africa_%28orthographic_projection%29.svg.png",
                 width=140)
st.sidebar.title("🌍 Filters")

regions_all = sorted(df_full["region"].unique().tolist())
sel_regions = st.sidebar.multiselect("Regions", regions_all, default=regions_all)

le_range = st.sidebar.slider(
    "Life Expectancy Range",
    float(df_full["life_expectancy"].min()),
    float(df_full["life_expectancy"].max()),
    (float(df_full["life_expectancy"].min()), float(df_full["life_expectancy"].max()))
)

gdp_range = st.sidebar.slider(
    "GDP per Capita Range (USD)",
    0, 15000, (0, 15000), step=500
)

show_raw = st.sidebar.checkbox("Show raw data table", value=False)

# Apply filters
df = df_full[
    df_full["region"].isin(sel_regions) &
    df_full["life_expectancy"].between(*le_range) &
    df_full["gdp_per_capita"].between(*gdp_range)
].copy()

# ── Header ───────────────────────────────────────────────────────────
st.title("🌍 Africa Health Indicators Dashboard")
st.markdown(f"""**Data:** Synthetic 2022 dataset | **Countries shown:** {len(df)} of {len(df_full)} | 
**Regions:** {\', \'.join(sel_regions)}""")
st.divider()

# ── KPI Metrics row ──────────────────────────────────────────────────
k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("🌡 Mean Life Exp.",  f"{df[\"life_expectancy\"].mean():.1f} yrs",
          delta=f"{df[\"life_expectancy\"].mean()-df_full[\"life_expectancy\"].mean():.1f} vs continent")
k2.metric("👶 Infant Mortality", f"{df[\"infant_mortality\"].mean():.1f} /1k")
k3.metric("💉 Vaccination",      f"{df[\"vaccination_dtp3\"].mean():.1f}%")
k4.metric("💧 Water Access",     f"{df[\"water_access\"].mean():.1f}%")
k5.metric("💵 Mean GDP/capita", f"${df[\"gdp_per_capita\"].mean():,.0f}")
st.divider()

# ── Tabs ─────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Overview", "🔬 Distributions", "🗺 Regional Analysis",
    "📈 Statistical Tests", "🏆 Rankings"
])

# ── TAB 1: Overview ──────────────────────────────────────────────────
with tab1:
    c1, c2 = st.columns([3, 2])

    with c1:
        st.subheader("GDP per Capita vs. Life Expectancy")
        x_var = st.selectbox("X axis", ["gdp_per_capita","health_expenditure",
                                          "vaccination_dtp3","water_access"], key="x1")
        y_var = st.selectbox("Y axis", ["life_expectancy","infant_mortality",
                                          "health_index","disease_burden"], key="y1")
        fig, ax = plt.subplots(figsize=(8,5))
        for region in sel_regions:
            sub = df[df["region"]==region]
            ax.scatter(sub[x_var], sub[y_var],
                       c=REGION_COLORS.get(region,"grey"),
                       label=region, s=55, alpha=0.8, edgecolors="white")
        if len(df) > 2:
            m,b = np.polyfit(df[x_var], df[y_var], 1)
            xl = np.linspace(df[x_var].min(), df[x_var].max(), 200)
            ax.plot(xl, m*xl+b, "--", color="#2C3E50", linewidth=1.5)
        ax.set_xlabel(x_var.replace("_"," ").title())
        ax.set_ylabel(y_var.replace("_"," ").title())
        ax.legend(fontsize=8, ncol=2)
        st.pyplot(fig)
        plt.close()

    with c2:
        st.subheader("Health Index Top 10")
        top10 = df.nlargest(10, "health_index")[["country","region","health_index"]]
        fig2, ax2 = plt.subplots(figsize=(5,5))
        colors_bar = [REGION_COLORS.get(r,"grey") for r in top10["region"]]
        ax2.barh(top10["country"], top10["health_index"],
                 color=colors_bar, alpha=0.85, edgecolor="white")
        ax2.set_xlabel("Health Index")
        st.pyplot(fig2)
        plt.close()

    if show_raw:
        with st.expander("Raw Data Table"):
            st.dataframe(df, use_container_width=True)

# ── TAB 2: Distributions ─────────────────────────────────────────────
with tab2:
    st.subheader("Distribution Analysis")
    col_to_plot = st.selectbox(
        "Select indicator",
        ["life_expectancy","infant_mortality","maternal_mortality",
         "gdp_per_capita","hiv_prevalence","vaccination_dtp3",
         "water_access","malaria_incidence"],
        key="dist_col"
    )
    apply_log = st.checkbox("Apply log transform", key="log_dist")
    apply_wins = st.checkbox("Apply 5% Winsorization", key="wins_dist")

    data = df[col_to_plot].dropna().values
    if apply_wins:
        data = np.array(winsorize(data, limits=[0.05,0.05]))
    if apply_log:
        data = np.log1p(data)

    fig3, axes3 = plt.subplots(1, 2, figsize=(12,4))
    axes3[0].hist(data, bins=20, color="#3498DB", alpha=0.75, edgecolor="white", density=True)
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(data)
    xk  = np.linspace(data.min(), data.max(), 200)
    axes3[0].plot(xk, kde(xk), "k-", linewidth=1.5)
    axes3[0].axvline(data.mean(),   color="red",   linestyle="--", label=f"mean={data.mean():.2f}")
    axes3[0].axvline(np.median(data), color="green", linestyle=":"  , label=f"median={np.median(data):.2f}")
    axes3[0].set_title(f"Histogram + KDE{\" (log)\" if apply_log else \"\"}")
    axes3[0].legend(fontsize=8)

    stats.probplot(data, dist="norm", plot=axes3[1])
    axes3[1].set_title("Q-Q Plot")
    axes3[1].get_lines()[0].set(markersize=4, alpha=0.6)

    sw_stat, sw_p = stats.shapiro(data)
    sk = stats.skew(data)
    st.pyplot(fig3)
    plt.close()

    st.info(f"""**Shapiro-Wilk:** W={sw_stat:.4f}  p={sw_p:.4f}  
→ Distribution is {\"approximately NORMAL\" if sw_p>0.05 else \"NOT NORMAL\"} at α=0.05  |  
**Skewness:** {sk:.3f}""")

# ── TAB 3: Regional Analysis ─────────────────────────────────────────
with tab3:
    st.subheader("Regional Comparison")
    metric = st.selectbox("Metric", ["life_expectancy","infant_mortality",
                                      "gdp_per_capita","vaccination_dtp3",
                                      "health_index","disease_burden"], key="reg_metric")
    chart_type = st.radio("Chart type", ["Box","Violin","Bar"], horizontal=True)

    region_order = df.groupby("region")[metric].median().sort_values().index.tolist()
    fig4, ax4 = plt.subplots(figsize=(10,5))

    if chart_type == "Box":
        sns.boxplot(data=df, x="region", y=metric, order=region_order,
                    palette=list(REGION_COLORS.values()), width=0.5, ax=ax4)
    elif chart_type == "Violin":
        sns.violinplot(data=df, x="region", y=metric, order=region_order,
                       palette=list(REGION_COLORS.values()), inner="quart", ax=ax4)
    else:
        sns.barplot(data=df, x="region", y=metric, order=region_order,
                    palette=list(REGION_COLORS.values()), errorbar="sd", ax=ax4)

    ax4.set_xlabel("")
    ax4.set_ylabel(metric.replace("_"," ").title())
    ax4.tick_params(axis="x", rotation=20)
    st.pyplot(fig4)
    plt.close()

    # Heatmap
    st.subheader("Regional Profile Heatmap")
    hm_cols = ["life_expectancy","infant_mortality","vaccination_dtp3",
               "water_access","health_index","disease_burden"]
    hm_cols = [c for c in hm_cols if c in df.columns]
    reg_means = df.groupby("region")[hm_cols].mean()
    reg_norm  = (reg_means - reg_means.min()) / (reg_means.max() - reg_means.min())
    fig5, ax5 = plt.subplots(figsize=(10,4))
    sns.heatmap(reg_norm, annot=reg_means.round(1), fmt=".1f",
                cmap="YlOrRd", ax=ax5, annot_kws={"size":9},
                linewidths=0.4)
    ax5.tick_params(axis="x",rotation=25,labelsize=9)
    st.pyplot(fig5)
    plt.close()

# ── TAB 4: Statistical Tests ─────────────────────────────────────────
with tab4:
    st.subheader("Hypothesis Testing")
    c_left, c_right = st.columns(2)

    with c_left:
        st.markdown("### Welch\'s Two-Sample t-Test")
        group_col = st.selectbox("Grouping variable", ["region","gdp_band"], key="grp")
        test_metric = st.selectbox("Metric", ["life_expectancy","infant_mortality",
                                               "gdp_per_capita","vaccination_dtp3"], key="tm")
        groups_avail = sorted(df[group_col].dropna().unique().tolist())
        g1_name = st.selectbox("Group 1", groups_avail, index=0, key="g1")
        g2_name = st.selectbox("Group 2", groups_avail, index=min(1,len(groups_avail)-1), key="g2")

        if st.button("Run t-Test"):
            g1 = df[df[group_col]==g1_name][test_metric].dropna()
            g2 = df[df[group_col]==g2_name][test_metric].dropna()
            if len(g1) > 1 and len(g2) > 1:
                t, p = stats.ttest_ind(g1, g2, equal_var=False)
                d    = (g1.mean()-g2.mean()) / np.sqrt((g1.std()**2+g2.std()**2)/2)
                st.success(f"""**t = {t:.4f}  |  p = {p:.4f}**
\n→ {\"Significant difference\" if p<0.05 else \"No significant difference\"} at α=0.05
\nCohen\'s d = {d:.3f} ({\"large\" if abs(d)>0.8 else \"medium\" if abs(d)>0.5 else \"small\"} effect)
\n{g1_name}: mean={g1.mean():.2f}, n={len(g1)}
\n{g2_name}: mean={g2.mean():.2f}, n={len(g2)}""")
            else:
                st.warning("Not enough data in one or both groups.")

    with c_right:
        st.markdown("### Pearson/Spearman Correlation")
        x_c = st.selectbox("X variable", ["gdp_per_capita","health_expenditure",
                                            "vaccination_dtp3","water_access"], key="xc")
        y_c = st.selectbox("Y variable", ["life_expectancy","infant_mortality",
                                            "health_index","disease_burden"], key="yc")
        if st.button("Run Correlation"):
            subset = df[[x_c, y_c]].dropna()
            r, rp  = stats.pearsonr(subset[x_c], subset[y_c])
            rho, rp2 = stats.spearmanr(subset[x_c], subset[y_c])
            st.success(f"""**Pearson r = {r:.4f}  p = {rp:.4f}**
\n**Spearman ρ = {rho:.4f}  p = {rp2:.4f}**
\nCorrelation strength: {\"Strong\" if abs(r)>0.7 else \"Moderate\" if abs(r)>0.4 else \"Weak\"}""")

            fig_c, ax_c = plt.subplots(figsize=(5,4))
            ax_c.scatter(subset[x_c], subset[y_c], alpha=0.7, s=50, edgecolors="white")
            m,b = np.polyfit(subset[x_c], subset[y_c], 1)
            xl = np.linspace(subset[x_c].min(), subset[x_c].max(), 200)
            ax_c.plot(xl, m*xl+b, "r--")
            ax_c.set_xlabel(x_c.replace("_"," ").title())
            ax_c.set_ylabel(y_c.replace("_"," ").title())
            st.pyplot(fig_c)
            plt.close()

# ── TAB 5: Rankings ──────────────────────────────────────────────────
with tab5:
    st.subheader("Country Rankings")
    rank_metric  = st.selectbox("Rank by", ["health_index","disease_burden",
                                              "life_expectancy","gdp_per_capita"], key="rank_m")
    ascending    = st.checkbox("Ascending (worst first)", value=False, key="asc")
    n_show       = st.slider("Number of countries", 5, 54, 20, key="n_rank")

    ranked = df.sort_values(rank_metric, ascending=ascending).head(n_show)
    ranked["rank"] = range(1, len(ranked)+1)

    fig_r, ax_r = plt.subplots(figsize=(10, max(4, n_show*0.32)))
    colors_r = [REGION_COLORS.get(r,"grey") for r in ranked["region"]]
    ax_r.barh(ranked["country"], ranked[rank_metric],
              color=colors_r, alpha=0.85, edgecolor="white")
    ax_r.invert_yaxis()
    ax_r.set_xlabel(rank_metric.replace("_"," ").title())
    ax_r.set_title(f"Top {n_show} countries by {rank_metric.replace(\'_\',' \').title()}")
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=r) for r,c in REGION_COLORS.items()
                       if r in ranked["region"].values]
    ax_r.legend(handles=legend_elements, fontsize=8, loc="lower right")
    st.pyplot(fig_r)
    plt.close()

    st.dataframe(
        ranked[["rank","country","region",rank_metric]].set_index("rank"),
        use_container_width=True
    )

# ── Footer ────────────────────────────────────────────────────────────
st.divider()
st.caption("MCS603 Programming II — Africa Health Dashboard | Data: Synthetic 2022 dataset")
'''

with open("dashboard.py", "w") as f:
    f.write(dashboard_code)

print("dashboard.py written successfully!")
print("\nTo launch:")
print("  pip install streamlit plotly seaborn scipy")
print("  streamlit run dashboard.py")
print("\nThe app will open at: http://localhost:8501")

---
## 6. Running & Deployment

### Run locally
```bash
streamlit run dashboard.py
# → opens http://localhost:8501 in your browser
```

### Configuration options
Create `.streamlit/config.toml` in your project folder:
```toml
[server]
port = 8501
headless = true

[theme]
primaryColor = "#003366"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F0F2F6"
textColor = "#262730"
font = "sans serif"
```

### Deploy to Streamlit Community Cloud (free)
1. Push `dashboard.py` and a `requirements.txt` to GitHub
2. Visit [share.streamlit.io](https://share.streamlit.io)
3. Connect your GitHub repo
4. Click **Deploy** — live URL in ~2 minutes

**`requirements.txt`:**
```
streamlit>=1.35
pandas>=2.0
numpy>=1.24
scipy>=1.10
matplotlib>=3.7
seaborn>=0.12
```

In [ ]:
requirements = """streamlit>=1.35
pandas>=2.0
numpy>=1.24
scipy>=1.10
matplotlib>=3.7
seaborn>=0.12
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt written.")

---
## 7. Exercises

### Exercise 1: Add a Data Upload Tab
Add a 6th tab to the dashboard called **"Upload & Analyse"**.  
The tab should allow the user to upload their own CSV file and automatically:
- Show a data profile (shape, dtypes, missing values)
- Display a correlation heatmap of numeric columns
- Offer a column selector for distribution analysis

In [ ]:
# Write the upload tab code here, then add it to dashboard.py
upload_tab_code = '''
# ── TAB 6: Upload & Analyse ─────────────────────────────────────────
with tab6:  # add tab6 to the st.tabs() call in dashboard.py
    st.subheader("Upload Your Own CSV")
    uploaded = st.file_uploader("Upload a CSV file", type="csv")
    
    if uploaded is not None:
        # TODO: load, profile, and visualise
        pass
    else:
        st.info("Upload a CSV file to begin analysis.")
'''
print(upload_tab_code)

### Exercise 2: Add a Download Button
In the Rankings tab, add a `st.download_button` that lets the user download the ranked table as a CSV file.

In [ ]:
# Sample pattern for download button
download_code = '''
import streamlit as st
import pandas as pd

# Assuming `ranked` is your DataFrame
csv = ranked.to_csv(index=False).encode("utf-8")

st.download_button(
    label="⬇ Download Rankings as CSV",
    data=csv,
    file_name="health_rankings.csv",
    mime="text/csv",
)
'''
print(download_code)

### Exercise 3: Session State Counter
Add a sidebar button that tracks how many times the user has run the t-test. Display the count using `st.session_state`.

In [ ]:
session_code = '''
import streamlit as st

# Initialise counter
if "test_count" not in st.session_state:
    st.session_state.test_count = 0

if st.button("Run t-Test"):
    st.session_state.test_count += 1
    # ... run test ...

st.sidebar.metric("Tests Run", st.session_state.test_count)
'''
print(session_code)

---
## Summary

| Concept | Streamlit API |
|---|---|
| Page config | `st.set_page_config(layout="wide")` |
| Text & titles | `st.title`, `st.header`, `st.markdown` |
| Metrics | `st.metric("label", value, delta)` |
| Widgets | `st.slider`, `st.selectbox`, `st.multiselect`, `st.checkbox` |
| Layout | `st.columns`, `st.tabs`, `st.expander`, `st.sidebar` |
| Charts | `st.pyplot(fig)`, `st.plotly_chart(fig)` |
| Data display | `st.dataframe(df)`, `st.table(df)` |
| Caching | `@st.cache_data` |
| State | `st.session_state` |
| Download | `st.download_button(data=csv)` |
| Deploy | `streamlit run dashboard.py` → Streamlit Cloud |

---
## Full Module Summary

| Notebook | Topic | Key Skills |
|---|---|---|
| 1 | SciPy Statistics | Normality tests, IQR, Winsorization, t-tests |
| 2 | Pandas Wrangling | Imputation, melt/pivot, GroupBy, feature engineering |
| 3 | Visualisation | Distributions, heatmaps, before/after, Seaborn |
| 4 | 8-Step Pipeline | End-to-end Africa health analysis |
| 5 | Streamlit | Interactive web dashboard |

> **Congratulations!** You have completed the MCS603 Data Analysis Module.